# بک‌تست ۶ ربات با داده OKX
سلول‌ها را به ترتیب اجرا کن (دکمه ▶ کنار هر سلول). داده‌ها در Google Drive ذخیره می‌شوند و دفعات بعد دانلود مجدد لازم نیست.

## ۱) اتصال به Drive و ساخت پوشه

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/okx_backtest', exist_ok=True)
%cd /content/drive/MyDrive/okx_backtest

## ۲) ساخت فایل‌ها (فقط یک‌بار لازم است)

In [ ]:
%%writefile crypto_signal_bots_lite.py
"""
ربات‌های سیگنال با تمرکز روی کیفیت بالا (Win Rate) — نسخه سبک v3
فقط ۶ ربات فعال نگه داشته شده‌اند:
- crypto-signal-bot (EMA Cross)، v60 (BTC Filter)، v70 (SMC Winner)
- v0a, v60a, v70a با مدیریت سرمایه و چرخه جبران
- ربات‌های غیرفعال (v10, v20, v30, v40, v50) به همراه استراتژی و منطقشان حذف شده‌اند
"""
import os
import json
import time
import requests
import numpy as np
import pandas as pd
import ccxt
from collections import Counter, defaultdict
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
import jdatetime
import excel_log

# =====================================================================
# تنظیمات عمومی
# =====================================================================
TELEGRAM_TOKEN = os.environ.get("TELEGRAM_TOKEN")
CHAT_ID = os.environ.get("CHAT_ID")
TOOBIT_API_KEY = os.environ.get("TOOBIT_API_KEY")
TOOBIT_SECRET = os.environ.get("TOOBIT_SECRET")

SYMBOLS = [
    "BTC-SWAP-USDT", "ETH-SWAP-USDT", "SOL-SWAP-USDT", "XRP-SWAP-USDT",
    "BNB-SWAP-USDT", "DOGE-SWAP-USDT", "ADA-SWAP-USDT", "TRX-SWAP-USDT",
    "AVAX-SWAP-USDT", "LINK-SWAP-USDT", "DOT-SWAP-USDT", "LTC-SWAP-USDT",
    "BCH-SWAP-USDT", "TON-SWAP-USDT", "SUI-SWAP-USDT", "NEAR-SWAP-USDT",
    "APT-SWAP-USDT", "ARB-SWAP-USDT", "OP-SWAP-USDT", "ATOM-SWAP-USDT",
    "UNI-SWAP-USDT", "AAVE-SWAP-USDT", "FIL-SWAP-USDT", "INJ-SWAP-USDT",
    "TIA-SWAP-USDT", "JUP-SWAP-USDT", "ENA-SWAP-USDT", "WIF-SWAP-USDT",
    "1000PEPE-SWAP-USDT", "1000SHIB-SWAP-USDT", "WLD-SWAP-USDT", "SEI-SWAP-USDT",
    "RUNE-SWAP-USDT", "LDO-SWAP-USDT", "STX-SWAP-USDT", "IMX-SWAP-USDT",
    "HBAR-SWAP-USDT", "ICP-SWAP-USDT", "ETC-SWAP-USDT", "XLM-SWAP-USDT",
    "CRV-SWAP-USDT", "SNX-SWAP-USDT", "COMP-SWAP-USDT", "SUSHI-SWAP-USDT",
    "1INCH-SWAP-USDT", "GRT-SWAP-USDT", "DYDX-SWAP-USDT", "GMX-SWAP-USDT",
    "PENDLE-SWAP-USDT", "ONDO-SWAP-USDT", "JTO-SWAP-USDT", "PYTH-SWAP-USDT",
    "STRK-SWAP-USDT", "MANTA-SWAP-USDT", "1000BONK-SWAP-USDT", "1000FLOKI-SWAP-USDT",
    "MEME-SWAP-USDT", "ORDI-SWAP-USDT", "1000SATS-SWAP-USDT", "1000RATS-SWAP-USDT",
    "POPCAT-SWAP-USDT", "BRETT-SWAP-USDT", "TURBO-SWAP-USDT", "1000000MOG-SWAP-USDT",
    "DOGS-SWAP-USDT", "FET-SWAP-USDT", "TAO-SWAP-USDT", "AKT-SWAP-USDT",
    "ILV-SWAP-USDT", "AXS-SWAP-USDT", "SAND-SWAP-USDT", "MANA-SWAP-USDT",
    "GALA-SWAP-USDT", "ALGO-SWAP-USDT", "VET-SWAP-USDT", "EGLD-SWAP-USDT",
    "FLOW-SWAP-USDT", "KAVA-SWAP-USDT", "ROSE-SWAP-USDT", "KAS-SWAP-USDT",
    "THETA-SWAP-USDT", "CHZ-SWAP-USDT", "ENJ-SWAP-USDT", "ZIL-SWAP-USDT",
    "ONE-SWAP-USDT", "QTUM-SWAP-USDT", "IOTA-SWAP-USDT", "NEO-SWAP-USDT"
]

BTC_SYMBOL = "BTC-SWAP-USDT"
MAX_HEARTBEATS = 2
MAX_SIGNALS_PER_RUN = 5
MAX_OPEN_SAME_DIR = 3
EXTRA_LOG_FIELDS = ["fvg_size_atr", "fvg_age_15m", "gap_pos", "last_candle_with_trade",
                    "sweep_depth_atr", "sweep_age_h", "asia_range_atr", "gap4_atr", "risk_atr"]
SUMMARY_FILE = "daily_summary.json"
SUMMARY_HOUR = 8

MIN_SL_PCT = 0.0035
MAX_ENTRY_DRIFT_ATR = 0.45

# =====================================================================
# تنظیمات مدیریت سرمایه (برای ربات‌های v0a, v60a, v70a)
# =====================================================================
INITIAL_BALANCE = 100.0   # بالانس اولیه هر ربات (دلار)
RISK_PERCENT = 3.0        # ریسک هر معامله (۳٪)
SL_PERCENT = 2.0          # حد ضرر (۲٪ از قیمت ورود)

TF_MS = {
    "5m": 5 * 60 * 1000,
    "15m": 15 * 60 * 1000,
    "1h": 60 * 60 * 1000,
    "4h": 4 * 60 * 60 * 1000,
    "1d": 24 * 60 * 60 * 1000,
}

# --- تنظیمات استراتژی‌ها (فقط استراتژی‌های فعال) ---
B1_CROSS_LOOKBACK = 6
B1_RSI_LONG = (45, 68)
B1_RSI_SHORT = (32, 55)
B1_VOLUME_MULT = 1.5
B1_SL_ATR = 1.6
B1_TP_ATR = 3.4
B1_MIN_SCORE = 0.70
B1_MIN_ADX_4H = 18
B1_MIN_ADX_1H = 16
B1_MIN_SLOPE_ATR = 0.08

B6_MIN_GAP_ATR = 0.28
B6_MIN_ADX = 20
B6_MAX_EXT_ATR = 6.0
B6_1H_ALIGN_BONUS = 0.12
B6_ALT_GAP_ATR = 0.22
B6_TOUCH_ATR = 0.22
B6_MAX_EXT_ALT_ATR = 1.0
B6_RSI_LONG = (40, 60)
B6_RSI_SHORT = (40, 60)
B6_VOLUME_MULT = 1.4
B6_SL_ATR, B6_TP_ATR = 1.4, 2.95
B6_MIN_SCORE = 0.65
B6_COOLDOWN_HOURS = 3

B8_SWEEP_LOOKBACK = 10
B8_TP_ATR_MULT = 2.0
B8_COOLDOWN_HOURS = 4
B8_SL_BUFFER_ATR = 0.2
B8_MIN_SL_ATR = 0.6
B8_MAX_SL_ATR = 3.0
B8_MAX_SL_PCT = 0.07
B8_MIN_RR = 1.2
B8_MIN_VOL_RATIO = 0.9
B8_RSI_MAX_LONG = 72
B8_RSI_MIN_SHORT = 28
B8_MIN_SCORE = 0.35
B8_MAX_SIGNALS_PER_RUN = 5

# =====================================================================
# Funnel
# =====================================================================
FUNNEL = defaultdict(Counter)
_CUR = {"bot": None}

def rej(label="unlabeled"):
    FUNNEL[_CUR["bot"]][label] += 1
    return None

def funnel_count(label):
    FUNNEL[_CUR["bot"]][label] += 1

def print_funnel(bot_name, top=12):
    c = FUNNEL.pop(bot_name, None)
    if not c:
        return
    print(f"📉 {bot_name} funnel:")
    for k, n in sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))[:top]:
        print(f"   {k}: {n}")

# =====================================================================
# ابزارهای عمومی
# =====================================================================
def get_iran_time():
    now = datetime.now(ZoneInfo("Asia/Tehran"))
    date_gregorian = now.strftime("%Y-%m-%d")
    now_jalali = jdatetime.datetime.fromgregorian(datetime=now)
    date_jalali = now_jalali.strftime("%Y/%m/%d")
    time_str = now.strftime("%H:%M:%S")
    return date_gregorian, date_jalali, time_str

def send_telegram(message, max_retries=3, timeout=20):
    if not TELEGRAM_TOKEN or not CHAT_ID:
        return None
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(url, json={"chat_id": CHAT_ID, "text": message}, timeout=timeout)
            if r.status_code == 200:
                return r.json().get("result", {}).get("message_id")
            print(f"⚠️ تلگرام status={r.status_code} (تلاش {attempt}/{max_retries})")
        except Exception as e:
            print(f"⚠️ خطا در ارسال تلگرام: {e}")
        if attempt < max_retries:
            time.sleep(2 * attempt)
    return None

def delete_telegram(message_id, max_retries=2, timeout=15):
    if not TELEGRAM_TOKEN or not CHAT_ID:
        return
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/deleteMessage"
    for attempt in range(1, max_retries + 1):
        try:
            requests.post(url, json={"chat_id": CHAT_ID, "message_id": message_id}, timeout=timeout)
            return
        except Exception:
            if attempt < max_retries:
                time.sleep(1)

def format_number(value):
    if value >= 1000: return f"{value:.2f}"
    elif value >= 1: return f"{value:.4f}"
    else: return f"{value:.6f}"

def load_json(path, default):
    try:
        with open(path, "r") as f: return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return default

def save_json(path, data):
    with open(path, "w") as f: json.dump(data, f)

def append_trade_csv(bot_name, record):
    import csv
    fname = f"trades_{bot_name.replace('crypto-signal-bot', 'bot')}.csv"
    file_exists = os.path.exists(fname)
    fields = ["bot", "symbol", "direction", "entry", "sl", "tp", "exit",
              "result", "pnl_r", "setup", "score", "rsi", "vol_ratio", "atr_pct",
              "risk_pct", "rr", "drift_atr", "btc_4h", "btc_1h", "open_same_dir",
              "position_size", "margin", "leverage", "profit_usd", "balance_after",
              "cycle_status"] + EXTRA_LOG_FIELDS + ["hold_hours", "opened_ms", "closed_ms"]
    if file_exists:
        try:
            with open(fname, "r", encoding="utf-8") as f:
                old_header = f.readline().strip().split(",")
            if old_header != fields:
                os.replace(fname, fname + ".old")
                file_exists = False
        except Exception as e:
            print(f"⚠️ CSV header check error: {e}")
    hold_hours = ""
    try:
        if record.get("opened_ms") and record.get("closed_ms"):
            hold_hours = round((record["closed_ms"] - record["opened_ms"]) / 3600000, 2)
    except Exception: pass
    risk = abs(float(record.get("entry", 0) or 0) - float(record.get("sl", 0) or 0))
    pnl_r = ""
    if risk > 0 and record.get("result") in ("TP", "SL"):
        pnl_r = round(abs(float(record["tp"]) - float(record["entry"])) / risk, 3) if record["result"] == "TP" else -1.0
    row = {
        "bot": bot_name, "symbol": record.get("symbol", ""), "direction": record.get("direction"),
        "entry": record.get("entry"), "sl": record.get("sl"), "tp": record.get("tp"),
        "exit": record.get("close_price"), "result": record.get("result"), "pnl_r": pnl_r,
        "setup": record.get("setup", ""), "score": record.get("score", ""),
        "rsi": record.get("rsi", ""), "vol_ratio": record.get("vol_ratio", ""),
        "atr_pct": record.get("atr_pct", ""), "risk_pct": record.get("risk_pct", ""),
        "rr": record.get("rr", ""), "drift_atr": record.get("drift_atr", ""),
        "btc_4h": record.get("btc_4h", ""), "btc_1h": record.get("btc_1h", ""),
        "open_same_dir": record.get("open_same_dir", ""), "position_size": record.get("position_size", ""),
        "margin": record.get("margin", ""), "leverage": record.get("leverage", ""),
        "profit_usd": record.get("profit_usd", ""), "balance_after": record.get("balance_after", ""),
        "cycle_status": record.get("cycle_status", ""),
        "hold_hours": hold_hours, "opened_ms": record.get("opened_ms", ""), "closed_ms": record.get("closed_ms", "")
    }
    for k in EXTRA_LOG_FIELDS: row[k] = record.get(k, "")
    try:
        with open(fname, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            if not file_exists: w.writeheader()
            w.writerow(row)
    except Exception as e:
        print(f"⚠️ CSV log error: {e}")

def now_ms(): return int(time.time() * 1000)

# =====================================================================
# Ctx و صرافی
# =====================================================================
class Ctx:
    def __init__(self, exchange, ref_time_ms=None):
        self.exchange = exchange
        self.cache = {}
        self.tickers = {}
        self.btc_filter = None
        self.ref_time_ms = ref_time_ms
    def now_utc_hour(self):
        if self.ref_time_ms is None: return datetime.now(timezone.utc).hour
        return datetime.fromtimestamp(self.ref_time_ms / 1000, tz=timezone.utc).hour

def create_exchange():
    params = {"enableRateLimit": True}
    if TOOBIT_API_KEY and TOOBIT_SECRET:
        params.update({"apiKey": TOOBIT_API_KEY, "secret": TOOBIT_SECRET})
    return ccxt.toobit(params)

def fetch_ohlcv(exchange, symbol, timeframe, limit=150):
    raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(raw, columns=["timestamp", "open", "high", "low", "close", "volume"])
    if len(df) and df["timestamp"].iloc[-1] + TF_MS[timeframe] > now_ms():
        df = df.iloc[:-1]
    return df.reset_index(drop=True)

def calculate_indicators(df):
    df["ema9"] = df["close"].ewm(span=9, adjust=False).mean()
    df["ema21"] = df["close"].ewm(span=21, adjust=False).mean()
    df["ema50"] = df["close"].ewm(span=50, adjust=False).mean()
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    rsi = rsi.where(~((avg_loss == 0) & (avg_gain > 0)), 100.0)
    rsi = rsi.where(~((avg_loss == 0) & (avg_gain == 0)), 50.0)
    df["rsi"] = rsi
    high_low = df["high"] - df["low"]
    high_close = (df["high"] - df["close"].shift()).abs()
    low_close = (df["low"] - df["close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["atr"] = tr.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    up = df["high"].diff()
    down = -df["low"].diff()
    plus_dm = up.where((up > down) & (up > 0), 0.0)
    minus_dm = down.where((down > up) & (down > 0), 0.0)
    atr_safe = df["atr"].replace(0, np.nan)
    plus_di = 100 * plus_dm.ewm(alpha=1/14, adjust=False, min_periods=14).mean() / atr_safe
    minus_di = 100 * minus_dm.ewm(alpha=1/14, adjust=False, min_periods=14).mean() / atr_safe
    di_sum = (plus_di + minus_di).replace(0, np.nan)
    dx = 100 * (plus_di - minus_di).abs() / di_sum
    df["adx"] = dx.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    bb_mid = df["close"].rolling(20).mean()
    bb_sd = df["close"].rolling(20).std()
    df["bb_mid"] = bb_mid
    df["bb_up"] = bb_mid + 2*bb_sd
    df["bb_lo"] = bb_mid - 2*bb_sd
    return df

def get_data(ctx, symbol, timeframe):
    key = (symbol, timeframe)
    if key not in ctx.cache:
        limit = 300 if timeframe == "1d" else 150
        df = fetch_ohlcv(ctx.exchange, symbol, timeframe, limit)
        if len(df) < 40:
            raise ValueError(f"داده کافی نیست: {symbol} {timeframe}")
        ctx.cache[key] = calculate_indicators(df)
    return ctx.cache[key]

def get_price(ctx, symbol):
    if ctx.ref_time_ms is not None:
        return float(get_data(ctx, symbol, "1h")["close"].iloc[-1])
    if symbol not in ctx.tickers:
        ctx.tickers[symbol] = float(ctx.exchange.fetch_ticker(symbol)["last"])
    return ctx.tickers[symbol]

def trend_of(df):
    last = df.iloc[-1]
    if last["ema9"] > last["ema21"]: return "LONG"
    if last["ema9"] < last["ema21"]: return "SHORT"
    return "NONE"

def get_btc_trend(ctx):
    try:
        return trend_of(get_data(ctx, BTC_SYMBOL, "4h")), trend_of(get_data(ctx, BTC_SYMBOL, "1h"))
    except Exception as e:
        print(f"⚠️ خطا در BTC: {e}")
        return "UNKNOWN", "UNKNOWN"

def atr_levels(direction, atr, sl_mult, tp_mult):
    def levels(entry):
        if direction == "LONG": return entry - atr*sl_mult, entry + atr*tp_mult
        return entry + atr*sl_mult, entry - atr*tp_mult
    return levels

def ratio(cur, avg): return cur / avg if avg and avg > 0 else 0.0

# =====================================================================
# استراتژی‌ها (فقط استراتژی‌های فعال: EMA Cross، BTC-aligned، SMC Winner)
# =====================================================================
def find_recent_cross(df, direction, lookback=B1_CROSS_LOOKBACK):
    if trend_of(df) != direction: return None, None
    n = len(df)
    for age in range(lookback):
        i = n - 1 - age
        if i < 1: break
        cur, prev = df.iloc[i], df.iloc[i-1]
        if direction == "LONG" and prev["ema9"] <= prev["ema21"] and cur["ema9"] > cur["ema21"]:
            return age, int(df["timestamp"].iloc[i])
        if direction == "SHORT" and prev["ema9"] >= prev["ema21"] and cur["ema9"] < cur["ema21"]:
            return age, int(df["timestamp"].iloc[i])
    return None, None

def analyze_cross(symbol, ctx):
    df_4h = get_data(ctx, symbol, "4h")
    direction, cross_age, cross_ts = None, None, None
    for d in ("LONG", "SHORT"):
        age, ts = find_recent_cross(df_4h, d, lookback=B1_CROSS_LOOKBACK)
        if age is not None: direction, cross_age, cross_ts = d, age, ts; break
    if direction is None: return rej("01_no_recent_4h_cross")
    last4 = df_4h.iloc[-1]
    atr_4h = float(last4["atr"])
    if np.isnan(atr_4h) or atr_4h <= 0: return rej("02_bad_atr_4h")
    adx_4h = float(last4["adx"])
    if np.isnan(adx_4h): return rej("03_bad_adx_4h")
    if adx_4h < B1_MIN_ADX_4H: return rej("04_adx_4h")
    ema9_4h, ema21_4h, ema50_4h = float(last4["ema9"]), float(last4["ema21"]), float(last4["ema50"])
    gap_4h = abs(ema9_4h - ema21_4h)
    if gap_4h < 0.35 * atr_4h: return rej("05_gap_4h")
    ema9_slope = ema9_4h - float(df_4h["ema9"].iloc[-4])
    slope_atr = abs(ema9_slope) / atr_4h
    if slope_atr < B1_MIN_SLOPE_ATR: return rej("06_slope_weak")
    if direction == "LONG" and ema9_slope <= 0: return rej("07_slope_direction")
    if direction == "SHORT" and ema9_slope >= 0: return rej("07_slope_direction")
    if direction == "LONG" and ema21_4h <= ema50_4h: return rej("08_ema21_vs_ema50")
    if direction == "SHORT" and ema21_4h >= ema50_4h: return rej("08_ema21_vs_ema50")
    df_1h = get_data(ctx, symbol, "1h")
    if trend_of(df_1h) != direction: return rej("09_trend_1h_mismatch")
    last_1h = df_1h.iloc[-1]
    atr_1h = float(last_1h["atr"])
    if np.isnan(atr_1h) or atr_1h <= 0: return rej("10_bad_atr_1h")
    adx_1h = float(last_1h["adx"])
    if np.isnan(adx_1h): return rej("11_bad_adx_1h")
    if adx_1h < B1_MIN_ADX_1H: return rej("12_adx_1h")
    gap_1h = abs(float(last_1h["ema9"]) - float(last_1h["ema21"]))
    if gap_1h < 0.40 * atr_1h: return rej("13_gap_1h")
    vol_1h = float(last_1h["volume"])
    avg_vol_1h = float(df_1h["volume"].iloc[-21:-1].mean())
    vol_ratio_1h = ratio(vol_1h, avg_vol_1h)
    if vol_ratio_1h < B1_VOLUME_MULT: return rej("14_volume_1h")
    df_15m = get_data(ctx, symbol, "15m")
    last = df_15m.iloc[-1]
    rsi = float(last["rsi"])
    if np.isnan(rsi): return rej("15_bad_rsi")
    lo, hi = (B1_RSI_LONG if direction == "LONG" else B1_RSI_SHORT)
    if not (lo <= rsi <= hi): return rej("16_rsi_15m")
    close = float(last["close"])
    if (atr_1h * B1_SL_ATR) / close < MIN_SL_PCT: return rej("17_min_sl_pct")
    trend1_score = min(gap_1h / atr_1h, 1.5) / 1.5
    trend4_score = min(gap_4h / atr_4h, 1.5) / 1.5
    vol_score = min(vol_ratio_1h, 3.0) / 3.0
    cross_score = (B1_CROSS_LOOKBACK - cross_age) / B1_CROSS_LOOKBACK
    rsi_score = min(abs(rsi - 50) / 20.0, 1.0)
    adx_score = min(adx_4h / 35.0, 1.0)
    score = (0.25*trend1_score + 0.25*trend4_score + 0.20*adx_score + 0.15*vol_score + 0.10*cross_score + 0.05*rsi_score)
    if score < B1_MIN_SCORE: return rej("18_score")
    return {"symbol": symbol, "direction": direction, "close": close, "atr": atr_1h,
            "rsi": rsi, "volume": vol_1h, "avg_volume": avg_vol_1h,
            "signal_key": cross_ts, "score": score,
            "setup": f"EMA Cross PRO (4H age {cross_age})",
            "levels": atr_levels(direction, atr_1h, B1_SL_ATR, B1_TP_ATR)}

def get_btc_filter(ctx):
    try:
        df_1d, df_4h, df_1h = get_data(ctx, BTC_SYMBOL, "1d"), get_data(ctx, BTC_SYMBOL, "4h"), get_data(ctx, BTC_SYMBOL, "1h")
    except Exception as e:
        print(f"⚠️ BTC filter data error: {e}")
        return None
    t_1d, t_4h, t_1h = trend_of(df_1d), trend_of(df_4h), trend_of(df_1h)
    print(f"🌍 BTC filter check: 1D={t_1d} | 4H={t_4h} | 1H={t_1h}")
    if t_1d not in ("LONG","SHORT") or t_4h not in ("LONG","SHORT"):
        print(f"🌍 BTC filter BLOCKED: trend unclear (1D={t_1d}, 4H={t_4h})")
        return None
    if t_1d != t_4h:
        print(f"🌍 BTC filter BLOCKED: 1D/4H conflict (1D={t_1d}, 4H={t_4h})")
        return None
    bias = t_1d
    last = df_4h.iloc[-1]
    atr, close = float(last["atr"]), float(last["close"])
    ema9, ema21 = float(last["ema9"]), float(last["ema21"])
    adx = float(last["adx"]) if not np.isnan(last["adx"]) else 0.0
    if np.isnan(atr) or atr <= 0:
        print(f"🌍 BTC filter BLOCKED: bad ATR ({atr})")
        return None
    gap_atr = abs(ema9 - ema21) / atr
    strength_ok = (gap_atr >= B6_MIN_GAP_ATR) or (adx >= B6_MIN_ADX)
    if not strength_ok:
        print(f"🌍 BTC filter BLOCKED: weak trend (gap={gap_atr:.2f} ATR, ADX={adx:.1f})")
        return None
    ext_atr = abs(close - ema21) / atr
    if ext_atr > B6_MAX_EXT_ATR:
        print(f"🌍 BTC filter BLOCKED: overextended ({ext_atr:.2f} ATR from EMA21)")
        return None
    align_1h = t_1h == bias
    print(f"🌍 BTC filter PASSED | bias={bias} | gap={gap_atr:.2f} ADX={adx:.1f} ext={ext_atr:.2f} | 1H_align={align_1h}")
    return {"bias":bias,"t_1d":t_1d,"t_4h":t_4h,"t_1h":t_1h,"align_1h":align_1h,"gap_atr":gap_atr,"adx":adx,"ext_atr":ext_atr}

def analyze_btc_aligned(symbol, ctx):
    btc = getattr(ctx, "btc_filter", None)
    if not btc:
        try:
            df_4h = get_data(ctx, symbol, "4h")
            bias = trend_of(df_4h)
            if bias not in ("LONG","SHORT"): return rej("01_no_bias")
        except Exception: return rej("01_btc_filter_blocked")
    else:
        bias = btc["bias"]
    df_4h = get_data(ctx, symbol, "4h")
    if trend_of(df_4h) != bias: return rej("02_symbol_4h_vs_btc_bias")
    l4 = df_4h.iloc[-1]
    atr_4h = float(l4["atr"])
    if np.isnan(atr_4h) or atr_4h <= 0: return rej("03_bad_atr_4h")
    gap_4h = abs(float(l4["ema9"]) - float(l4["ema21"]))
    if gap_4h < B6_ALT_GAP_ATR * atr_4h: return rej("04_gap_4h")
    df_1h = get_data(ctx, symbol, "1h")
    last = df_1h.iloc[-1]
    close, open_, high, low = (float(last[k]) for k in ("close","open","high","low"))
    atr, rsi, ema21 = float(last["atr"]), float(last["rsi"]), float(last["ema21"])
    if np.isnan(atr) or np.isnan(rsi) or atr <= 0: return rej("05_bad_atr_rsi_1h")
    if bias == "LONG":
        touched = (low <= ema21 + B6_TOUCH_ATR*atr and close > ema21 and close > open_ and close - ema21 <= B6_MAX_EXT_ALT_ATR*atr)
        rsi_ok = B6_RSI_LONG[0] <= rsi <= B6_RSI_LONG[1]
    else:
        touched = (high >= ema21 - B6_TOUCH_ATR*atr and close < ema21 and close < open_ and ema21 - close <= B6_MAX_EXT_ALT_ATR*atr)
        rsi_ok = B6_RSI_SHORT[0] <= rsi <= B6_RSI_SHORT[1]
    volume, avg_volume = float(last["volume"]), float(df_1h["volume"].iloc[-21:-1].mean())
    vol_ratio = ratio(volume, avg_volume)
    vol_ok = vol_ratio >= B6_VOLUME_MULT
    if not (touched and rsi_ok and vol_ok):
        tags = []
        if not touched: tags.append("pullback")
        if not rsi_ok: tags.append("rsi")
        if not vol_ok: tags.append("vol")
        return rej("06_filters_" + "+".join(tags))
    if (atr * B6_SL_ATR) / close < MIN_SL_PCT: return rej("07_min_sl_pct")
    trend_score = min(gap_4h/atr_4h, 1.2)/1.2
    vol_score = min(vol_ratio/2.5, 1.0)
    btc_score = 0.55 + (B6_1H_ALIGN_BONUS if btc and btc.get("align_1h") else 0.0)
    if btc: btc_score += 0.15 * min(btc["gap_atr"]/0.6, 1.0)
    score = 0.35*btc_score + 0.30*trend_score + 0.20*vol_score + 0.15*min(abs(rsi-50)/20.0,1.0)
    if score < B6_MIN_SCORE: return rej("08_score")
    return {"symbol": symbol, "direction": bias, "close": close, "atr": atr, "rsi": rsi,
            "volume": volume, "avg_volume": avg_volume, "signal_key": int(last["timestamp"]),
            "score": score, "setup": f"BTC-aligned pullback 1H (BTC {bias})",
            "levels": atr_levels(bias, atr, B6_SL_ATR, B6_TP_ATR)}

def analyze_smc_winner(symbol, ctx):
    try: df4 = get_data(ctx, symbol, "4h")
    except Exception: return rej("01_no_4h_data")
    if len(df4) < 60: return rej("02_4h_short_data")
    curr = df4.iloc[-1]
    atr = float(curr["atr"])
    if np.isnan(atr) or atr <= 0: return rej("03_bad_atr")
    close, open_, low, high = float(curr["close"]), float(curr["open"]), float(curr["low"]), float(curr["high"])
    if close <= 0: return rej("04_bad_price")
    rsi = float(curr["rsi"]) if not np.isnan(curr["rsi"]) else 50.0
    ema21, ema50 = float(curr["ema21"]), float(curr["ema50"])
    prev_low = float(df4["low"].iloc[-B8_SWEEP_LOOKBACK-1:-1].min())
    prev_high = float(df4["high"].iloc[-B8_SWEEP_LOOKBACK-1:-1].max())
    if low < prev_low and close > prev_low and close > open_:
        direction = "LONG"
        if not (ema21 > ema50 and close > ema50): return rej("06_trend_filter")
        wick = min(open_, close) - low
        sl_dist = close - (low - atr * B8_SL_BUFFER_ATR)
    elif high > prev_high and close < prev_high and close < open_:
        direction = "SHORT"
        if not (ema21 < ema50 and close < ema50): return rej("06_trend_filter")
        wick = high - max(open_, close)
        sl_dist = (high + atr * B8_SL_BUFFER_ATR) - close
    else: return rej("05_no_sweep")
    if sl_dist <= 0: return rej("07_sl_dist_invalid")
    sl_dist = max(sl_dist, atr * B8_MIN_SL_ATR)
    if sl_dist > atr * B8_MAX_SL_ATR or sl_dist / close > B8_MAX_SL_PCT: return rej("08_sl_too_far")
    if sl_dist / close < MIN_SL_PCT: return rej("09_min_sl_pct")
    tp_dist = atr * B8_TP_ATR_MULT
    rr = tp_dist / sl_dist
    if rr < B8_MIN_RR: return rej("10_rr_low")
    volume, avg_volume = float(curr["volume"]), float(df4["volume"].iloc[-21:-1].mean())
    vol_ratio = ratio(volume, avg_volume)
    if vol_ratio < B8_MIN_VOL_RATIO: return rej("11_volume")
    if direction == "LONG" and rsi > B8_RSI_MAX_LONG: return rej("12_rsi_extreme")
    if direction == "SHORT" and rsi < B8_RSI_MIN_SHORT: return rej("12_rsi_extreme")
    trend_score = min(abs(ema21 - ema50)/atr, 1.0)
    vol_score = min(vol_ratio/2.0, 1.0)
    wick_score = min(wick/atr, 1.0)
    rr_score = min(rr/2.5, 1.0)
    score = 0.30*trend_score + 0.25*vol_score + 0.25*wick_score + 0.20*rr_score
    if score < B8_MIN_SCORE: return rej("13_score")
    def levels(entry):
        if direction == "LONG": return entry - sl_dist, entry + tp_dist
        return entry + sl_dist, entry - tp_dist
    sweep_depth = abs(prev_low - low) if direction == "LONG" else abs(high - prev_high)
    meta = {"fvg_size_atr":"","fvg_age_15m":"","gap_pos":"",
            "last_candle_with_trade":1 if ((direction=="LONG" and close>open_) or (direction=="SHORT" and close<open_)) else 0,
            "sweep_depth_atr":round(sweep_depth/atr,3),"sweep_age_h":0,
            "asia_range_atr":"","gap4_atr":"","risk_atr":round(sl_dist/atr,3)}
    return {"symbol": symbol, "direction": direction, "close": close, "atr": atr, "rsi": rsi,
            "volume": volume, "avg_volume": avg_volume, "signal_key": int(curr["timestamp"]),
            "score": score, "setup": f"SMC Winner ({direction} Sweep, RR {rr:.1f})",
            "levels": levels, "meta": meta}

# =====================================================================
# مدیریت سرمایه و چرخه جبران
# =====================================================================
def calculate_position_sizing(bot, entry, sl, tp, balance, in_recovery, recovery_losses):
    sl_dist_pct = abs(entry - sl) / entry if entry > 0 else 0
    rr_ratio = abs(tp - entry) / abs(entry - sl) if abs(entry - sl) > 0 else 2.0
    if not in_recovery:
        risk_amount = balance * (bot.risk_percent / 100.0)
    else:
        total_loss = sum(recovery_losses)
        risk_amount = total_loss / rr_ratio if rr_ratio > 0 else 0
    position_value = risk_amount / sl_dist_pct if sl_dist_pct > 0 else 0
    leverage_needed = position_value / balance if balance > 0 else 1.0
    position_size = position_value / entry if entry > 0 else 0
    return {"position_size": round(position_size, 6), "position_value": round(position_value, 2),
            "leverage": round(max(1.0, leverage_needed), 2), "risk_amount": round(risk_amount, 2)}

def update_open_trades_with_sizing(bot, exchange, signals):
    updated = False
    for symbol, data in list(signals.items()):
        if symbol.startswith("__"): continue
        if data.get("status") == "OPEN":
            try: result, close_price = check_trade_result(exchange, symbol, data)
            except Exception as e:
                print(f"⚠️ خطا در بررسی {symbol}: {e}"); continue
            if result:
                data["status"] = "CLOSED"
                data["result"] = result
                data["closed_ms"] = now_ms()
                data["close_price"] = float(close_price)
                data["close_telegram_sent"] = False
                data["close_msg_attempts"] = 0
                data["symbol"] = symbol
                entry_price = data["entry"]
                pos_size = data.get("position_size", 0)
                if result == "TP":
                    profit_usd = abs(data["tp"] - entry_price) * pos_size
                    new_balance = signals.get("__balance__", bot.initial_balance) + profit_usd
                    signals["__balance__"] = new_balance
                    signals["__in_recovery__"] = False
                    signals["__recovery_losses__"] = []
                    data["cycle_status"] = "ENDED (Win)"
                else:
                    loss_usd = abs(entry_price - data["sl"]) * pos_size
                    new_balance = signals.get("__balance__", bot.initial_balance) - loss_usd
                    signals["__balance__"] = new_balance
                    losses = signals.get("__recovery_losses__", [])
                    losses.append(loss_usd)
                    signals["__recovery_losses__"] = losses
                    signals["__in_recovery__"] = True
                    data["cycle_status"] = f"IN RECOVERY (Loss #{len(losses)})"
                data["profit_usd"] = round(profit_usd if result == "TP" else -loss_usd, 2)
                data["balance_after"] = round(new_balance, 2)
                updated = True
                append_trade_csv(bot.name, data)
                excel_log.log_result(bot.name, symbol, result, data["closed_ms"])
                print(f"{'🎯' if result == 'TP' else '🛑'} {symbol} {data['direction']} hit {result} at {close_price}")
        if (data.get("status") == "CLOSED" and data.get("close_telegram_sent") is False
                and data.get("close_msg_attempts", 0) < CLOSE_MSG_MAX_ATTEMPTS):
            msg_id = send_telegram(build_close_message(
                bot, symbol, data["direction"], data["result"], data["close_price"], data["entry"],
                profit_usd=data.get("profit_usd"), balance_after=data.get("balance_after"),
                cycle_status=data.get("cycle_status")))
            data["close_msg_attempts"] = data.get("close_msg_attempts", 0) + 1
            data["close_telegram_sent"] = bool(msg_id)
            updated = True
    return updated

# =====================================================================
# توابع مدیریت معاملات
# =====================================================================
CLOSE_MSG_MAX_ATTEMPTS = 5

def check_trade_result(exchange, symbol, data):
    direction, tp, sl = data["direction"], data["tp"], data["sl"]
    opened_ms = data.get("opened_ms")
    if opened_ms:
        try: candles = exchange.fetch_ohlcv(symbol, timeframe="5m", since=int(opened_ms), limit=1000)
        except Exception: candles = []
        for cd in candles:
            if cd[0] < opened_ms: continue
            high, low, close = cd[2], cd[3], cd[4]
            if direction == "LONG": hit_sl, hit_tp = low <= sl, high >= tp
            else: hit_sl, hit_tp = high >= sl, low <= tp
            if hit_sl and hit_tp:
                dist_tp = abs(close - tp); dist_sl = abs(close - sl)
                return ("TP", tp) if dist_tp < dist_sl else ("SL", sl)
            if hit_sl: return "SL", sl
            if hit_tp: return "TP", tp
    price = float(exchange.fetch_ticker(symbol)["last"])
    if direction == "LONG":
        if price <= sl: return "SL", price
        if price >= tp: return "TP", price
    else:
        if price >= sl: return "SL", price
        if price <= tp: return "TP", price
    return None, None

def build_close_message(bot, symbol, direction, result, close_price, entry, profit_usd=None, balance_after=None, cycle_status=None):
    date_gregorian, date_jalali, time_str = get_iran_time()
    emoji = "🎯" if result == "TP" else "🛑"
    title = "حد سود (TP) لمس شد" if result == "TP" else "حد ضرر (SL) لمس شد"
    fin_lines = ""
    if profit_usd is not None:
        fin_lines = f"""
💰 سود/ضرر: ${profit_usd:+.2f}
💵 موجودی جدید: ${balance_after:.2f}
🔄 وضعیت چرخه: {cycle_status}
"""
    return f"""{emoji} {bot.title_tag} بسته شدن معامله
━━━━━━━━━━━━━━━━━
📌 نماد: {symbol}
📊 جهت: {direction}
🏁 نتیجه: {title}
💰 قیمت بسته شدن: {format_number(close_price)}
🚪 قیمت ورود: {format_number(entry)}
{fin_lines}
━━━━━━━━━━━━━━━━━
📅 تاریخ شمسی: {date_jalali}
📅 تاریخ میلادی: {date_gregorian}
⏰ ساعت: {time_str}
━━━━━━━━━━━━━━━━━"""

def build_signal_message(bot, c, entry, sl, tp, rr, sizing=None):
    date_gregorian, date_jalali, time_str = get_iran_time()
    emoji = "🟢" if c["direction"] == "LONG" else "🔴"
    fin_lines = ""
    if sizing:
        fin_lines = f"""
📦 حجم پوزیشن: {sizing['position_size']}
💵 ارزش دلاری: ${sizing['position_value']}
⚙️ اهرم: {sizing['leverage']}x
"""
    return f"""{emoji} {bot.title_tag} سیگنال {c['direction']}
━━━━━━━━━━━━━━━━━
📌 نماد: {c['symbol']}
🔎 ستاپ: {c['setup']}
💰 قیمت ورود: {format_number(entry)}
🎯 حد سود (TP): {format_number(tp)}
🛑 حد ضرر (SL): {format_number(sl)}
📊 نسبت R:R: 1:{rr:.1f}
{fin_lines}
━━━━━━━━━━━━━━━━━
📅 تاریخ شمسی: {date_jalali}
📅 تاریخ میلادی: {date_gregorian}
⏰ ساعت: {time_str}
━━━━━━━━━━━━━━━━━"""

# =====================================================================
# update_open_trades (برای ربات‌های بدون مدیریت سرمایه)
# =====================================================================
def update_open_trades(bot, exchange):
    signals = load_json(bot.signals_file, {})
    updated = False
    for symbol, data in list(signals.items()):
        if symbol.startswith("__"): continue
        if data.get("status") == "OPEN":
            try: result, close_price = check_trade_result(exchange, symbol, data)
            except Exception as e:
                print(f"⚠️ خطا در بررسی {symbol}: {e}"); continue
            if result:
                data["status"] = "CLOSED"
                data["result"] = result
                data["closed_ms"] = now_ms()
                data["close_price"] = float(close_price)
                data["close_telegram_sent"] = False
                data["close_msg_attempts"] = 0
                data["symbol"] = symbol
                updated = True
                append_trade_csv(bot.name, data)
                excel_log.log_result(bot.name, symbol, result, data["closed_ms"])
                print(f"{'🎯' if result == 'TP' else '🛑'} {symbol} {data['direction']} hit {result} at {close_price}")
        if (data.get("status") == "CLOSED" and data.get("close_telegram_sent") is False
                and data.get("close_msg_attempts", 0) < CLOSE_MSG_MAX_ATTEMPTS):
            msg_id = send_telegram(build_close_message(
                bot, symbol, data["direction"], data["result"], data["close_price"], data["entry"]))
            data["close_msg_attempts"] = data.get("close_msg_attempts", 0) + 1
            data["close_telegram_sent"] = bool(msg_id)
            updated = True
    if updated: save_json(bot.signals_file, signals)

# =====================================================================
# Bot
# =====================================================================
class Bot:
    def __init__(self, name, label, analyze, heartbeat_file, signals_file,
                 cooldown_hours=0, max_signals=MAX_SIGNALS_PER_RUN,
                 btc_filter_4h=True, btc_filter_1h=False, live=True,
                 single_trade=False, has_sizing=False,
                 initial_balance=INITIAL_BALANCE, risk_percent=RISK_PERCENT, sl_percent=SL_PERCENT):
        self.name = name
        self.label = label
        self.analyze = analyze
        self.heartbeat_file = heartbeat_file
        self.signals_file = signals_file
        self.cooldown_ms = int(cooldown_hours * 3600 * 1000)
        # ربات‌های تک‌معامله هیچ‌وقت نباید در یک اجرا بیش از یک سیگنال بدهند
        self.max_signals = 1 if single_trade else max_signals
        self.btc_filter_4h = btc_filter_4h
        self.btc_filter_1h = btc_filter_1h
        self.live = live
        self.single_trade = single_trade
        self.has_sizing = has_sizing
        self.initial_balance = initial_balance
        self.risk_percent = risk_percent
        self.sl_percent = sl_percent
    @property
    def title_tag(self):
        return f"[{self.name}]" + (f" ({self.label})" if self.label else "")

# =====================================================================
# BOTS نهایی (۶ ربات فعال)
# =====================================================================
BOTS = [
    # --- ۳ ربات اصلی ---
    Bot("crypto-signal-bot", "EMA Cross", analyze_cross,
        "server_messages.json", "last_signals.json", live=True),
    Bot("crypto-signal-bot-v60", "BTC Filter", analyze_btc_aligned,
        "server_messages_v60.json", "last_signals_v60.json",
        cooldown_hours=B6_COOLDOWN_HOURS, btc_filter_4h=False, live=True),
    Bot("crypto-signal-bot-v70", "SMC Winner", analyze_smc_winner,
        "server_messages_v70.json", "last_signals_v70.json",
        cooldown_hours=B8_COOLDOWN_HOURS, max_signals=B8_MAX_SIGNALS_PER_RUN,
        btc_filter_4h=True, live=True),
    # --- ۳ ربات با مدیریت سرمایه (تک‌معامله) ---
    Bot("crypto-signal-bot-v0a", "EMA Cross (Managed)", analyze_cross,
        "server_messages_v0a.json", "last_signals_v0a.json",
        live=True, single_trade=True, has_sizing=True,
        initial_balance=100.0, risk_percent=3.0, sl_percent=2.0),
    Bot("crypto-signal-bot-v60a", "BTC Filter (Managed)", analyze_btc_aligned,
        "server_messages_v60a.json", "last_signals_v60a.json",
        cooldown_hours=B6_COOLDOWN_HOURS, btc_filter_4h=False,
        live=True, single_trade=True, has_sizing=True,
        initial_balance=100.0, risk_percent=3.0, sl_percent=2.0),
    Bot("crypto-signal-bot-v70a", "SMC Winner (Managed)", analyze_smc_winner,
        "server_messages_v70a.json", "last_signals_v70a.json",
        cooldown_hours=B8_COOLDOWN_HOURS, max_signals=1,
        btc_filter_4h=True, live=True, single_trade=True, has_sizing=True,
        initial_balance=100.0, risk_percent=3.0, sl_percent=2.0),
]
LIVE_BOTS = [b for b in BOTS if b.live]

# =====================================================================
# Heartbeat
# =====================================================================
HEARTBEAT_FILE = "server_messages.json"
def send_heartbeat_all(results, btc_4h, btc_1h, warning=None):
    date_gregorian, date_jalali, time_str = get_iran_time()
    for bot in LIVE_BOTS[1:]:
        old = load_json(bot.heartbeat_file, {}).get("message_ids", [])
        for mid in old: delete_telegram(mid)
        save_json(bot.heartbeat_file, {"message_ids": []})
    old_main_ids = load_json(HEARTBEAT_FILE, {}).get("message_ids", [])
    for mid in old_main_ids: delete_telegram(mid)
    lines, ok = [], 0
    for bot in LIVE_BOTS:
        r = results.get(bot.name, "ERROR")
        if r == "ERROR": lines.append(f"❌ {bot.name}")
        else:
            ok += 1
            lines.append(f"✅ {bot.name}" + (f"  ({r} سیگنال جدید)" if r else ""))
    extra = f"\n⚠️ {warning}" if warning else ""
    msg = f"""🖥️ سیستم آنلاین | {ok}/{len(LIVE_BOTS)} ربات سالم{extra}
━━━━━━━━━━━━━━━━━
{chr(10).join(lines)}
━━━━━━━━━━━━━━━━━
🌍 BTC: 4H={btc_4h} | 1H={btc_1h}
📅 تاریخ شمسی: {date_jalali}
📅 تاریخ میلادی: {date_gregorian}
⏰ ساعت: {time_str}
━━━━━━━━━━━━━━━━━"""
    new_id = send_telegram(msg)
    if new_id: save_json(HEARTBEAT_FILE, {"message_ids": [new_id]})

# =====================================================================
# خلاصه روزانه
# =====================================================================
def trade_r(rec):
    if rec.get("status") != "CLOSED": return None
    risk = abs(rec["entry"] - rec["sl"])
    if risk <= 0: return None
    return abs(rec["tp"] - rec["entry"]) / risk if rec.get("result") == "TP" else -1.0

def maybe_send_daily_summary():
    now = datetime.now(ZoneInfo("Asia/Tehran"))
    today = now.strftime("%Y-%m-%d")
    state = load_json(SUMMARY_FILE, {})
    if now.hour < SUMMARY_HOUR or state.get("date") == today: return
    date_gregorian, date_jalali, time_str = get_iran_time()
    lines = []
    for bot in LIVE_BOTS:
        recs = [r for k, r in load_json(bot.signals_file, {}).items() if not k.startswith("__")]
        closed = [r for r in recs if r.get("status") == "CLOSED"]
        opened = [r for r in recs if r.get("status") == "OPEN"]
        rs = [x for x in (trade_r(r) for r in closed) if x is not None]
        if rs:
            wins = sum(1 for r in closed if r.get("result") == "TP")
            lines.append(f"🤖 {bot.name}\n   بسته‌شده: {len(rs)} | برد: {wins} ({wins/len(rs):.0%}) | R کل: {sum(rs):+.1f} | باز: {len(opened)}")
        else:
            lines.append(f"🤖 {bot.name}\n   هنوز معامله‌ی بسته‌شده‌ای نیست | باز: {len(opened)}")
    msg = ("📊 خلاصه‌ی روزانه‌ی ربات‌ها\n━━━━━━━━━━━━━━━━━\n" + "\n".join(lines) +
           f"\n━━━━━━━━━━━━━━━━━\n📅 تاریخ شمسی: {date_jalali}\n📅 تاریخ میلادی: {date_gregorian}\n⏰ ساعت: {time_str}\n━━━━━━━━━━━━━━━━━")
    if send_telegram(msg): save_json(SUMMARY_FILE, {"date": today})

# =====================================================================
# چرخه ربات (با مدیریت سرمایه)
# =====================================================================
def run_bot(bot, ctx, btc_4h, btc_1h):
    _CUR["bot"] = bot.name
    print("=" * 50)
    print(f"▶ {bot.name} {('(' + bot.label + ')') if bot.label else ''}")
    signals = load_json(bot.signals_file, {})
    # ابتدا وضعیت معاملات باز بررسی شود (قبل از چک تک‌معامله)
    if bot.has_sizing:
        if update_open_trades_with_sizing(bot, ctx.exchange, signals):
            save_json(bot.signals_file, signals)
    else:
        update_open_trades(bot, ctx.exchange)
        signals = load_json(bot.signals_file, {})
    if bot.single_trade:
        open_trades = [s for k, s in signals.items() if not k.startswith("__") and s.get("status") == "OPEN"]
        if open_trades:
            print(f"⏳ {bot.name} has an OPEN trade. Skipping this cycle.")
            return 0
    candidates = []
    t_now = now_ms()
    for symbol in SYMBOLS:
        try:
            prev = signals.get(symbol)
            if prev and prev.get("status") == "OPEN": funnel_count("~skip_open_trade"); continue
            if (prev and bot.cooldown_ms and prev.get("closed_ms") and t_now - prev["closed_ms"] < bot.cooldown_ms):
                funnel_count("~skip_cooldown"); continue
            funnel_count("0_analyzed")
            c = bot.analyze(symbol, ctx)
            if c is None: continue
            funnel_count("PASSED_analyze")
            prev_key = (prev.get("signal_key", prev.get("cross_ts")) if prev else None)
            if prev and prev_key == c["signal_key"] and prev.get("direction") == c["direction"]:
                funnel_count("~already_signaled"); continue
            if symbol != BTC_SYMBOL:
                if bot.btc_filter_4h and btc_4h in ("LONG","SHORT") and c["direction"] != btc_4h:
                    funnel_count("~btc_4h_filter"); continue
                if bot.btc_filter_1h and btc_1h in ("LONG","SHORT") and c["direction"] != btc_1h:
                    funnel_count("~btc_1h_filter"); continue
            candidates.append(c)
        except Exception as e:
            funnel_count("EXCEPTION")
            print(f"⚠️ خطا در {symbol}: {e}")
            continue
    candidates.sort(key=lambda x: x["score"], reverse=True)
    sent = 0
    open_dir = Counter(s.get("direction") for k, s in signals.items() if not k.startswith("__") and s.get("status") == "OPEN")
    for c in candidates:
        if sent >= bot.max_signals: break
        symbol = c["symbol"]
        try:
            if open_dir[c["direction"]] >= MAX_OPEN_SAME_DIR: funnel_count("~cap_same_direction"); continue
            entry = get_price(ctx, symbol)
            if abs(entry - c["close"]) > c["atr"] * MAX_ENTRY_DRIFT_ATR:
                funnel_count("~send_price_drift"); continue
            lv = c["levels"](entry)
            if lv is None: funnel_count("~send_levels_none"); continue
            sl, tp = lv
            if sl <= 0 or tp <= 0: funnel_count("~send_invalid_sl_tp"); continue
            rr = abs(tp - entry) / abs(entry - sl)
            sizing = None
            if bot.has_sizing:
                balance = signals.get("__balance__", bot.initial_balance)
                in_recovery = signals.get("__in_recovery__", False)
                recovery_losses = signals.get("__recovery_losses__", [])
                sizing = calculate_position_sizing(bot, entry, sl, tp, balance, in_recovery, recovery_losses)
            msg_id = send_telegram(build_signal_message(bot, c, entry, sl, tp, rr, sizing))
            opened_ts = ctx.ref_time_ms if ctx.ref_time_ms is not None else now_ms()
            record = {
                "direction": c["direction"], "entry": entry, "tp": tp, "sl": sl,
                "signal_key": c["signal_key"], "setup": c["setup"],
                "score": round(c["score"], 4), "rsi": round(c["rsi"], 2),
                "vol_ratio": round(ratio(c["volume"], c["avg_volume"]), 2),
                "atr_pct": round(c["atr"] / entry * 100, 3),
                "risk_pct": round(abs(entry - sl) / entry * 100, 3),
                "rr": round(rr, 2),
                "drift_atr": round(abs(entry - c["close"]) / c["atr"], 3),
                "btc_4h": btc_4h, "btc_1h": btc_1h,
                "open_same_dir": open_dir[c["direction"]],
                **c.get("meta", {}),
                "time": datetime.now(timezone.utc).isoformat(),
                "telegram_sent": bool(msg_id), "telegram_message_id": msg_id,
            }
            if sizing:
                record.update({"position_size": sizing["position_size"], "position_value": sizing["position_value"],
                               "leverage": sizing["leverage"], "risk_amount": sizing["risk_amount"]})
            if msg_id:
                record["status"] = "OPEN"
                record["opened_ms"] = opened_ts
                sent += 1
                open_dir[c["direction"]] += 1
                funnel_count("SENT_signal")
                excel_log.log_signal(bot.name, symbol, c["direction"], entry, tp, sl)
            else:
                record["status"] = "NOT_SENT"
                funnel_count("~telegram_not_sent")
            signals[symbol] = record
            save_json(bot.signals_file, signals)
        except Exception as e:
            funnel_count("EXCEPTION_send")
            print(f"⚠️ خطا در ارسال {symbol}: {e}")
            continue
    print_funnel(bot.name)
    return sent

# =====================================================================
# اجرای اصلی
# =====================================================================
if __name__ == "__main__":
    print("=" * 50)
    print("Starting multi-bot cycle (Quality Mode): " + ", ".join(b.name for b in LIVE_BOTS))
    exchange = create_exchange()
    ctx = Ctx(exchange)
    btc_4h, btc_1h = get_btc_trend(ctx)
    try: btc_1d = trend_of(get_data(ctx, BTC_SYMBOL, "1d"))
    except Exception: btc_1d = "UNKNOWN"
    print(f"🌍 BTC Trend: 1D={btc_1d} | 4H={btc_4h} | 1H={btc_1h}")
    warning = "دریافت داده‌ی BTC ناموفق بود" if btc_4h == "UNKNOWN" else None
    ctx.btc_filter = get_btc_filter(ctx)
    if not ctx.btc_filter: print("🌍 BTC filter blocked — v60 will skip all symbols this cycle.")
    results = {}
    for bot in LIVE_BOTS:
        try: results[bot.name] = run_bot(bot, ctx, btc_4h, btc_1h)
        except Exception as e:
            results[bot.name] = "ERROR"
            print(f"⚠️ خطای کلی در {bot.name}: {e}")
    try: send_heartbeat_all(results, btc_4h, btc_1h, warning)
    except Exception as e: print(f"⚠️ خطا در heartbeat: {e}")
    try: maybe_send_daily_summary()
    except Exception as e: print(f"⚠️ خطا در خلاصه‌ی روزانه: {e}")
    print("All bots complete.")
    print("=" * 50)


In [ ]:
%%writefile download_okx_data.py
#!/usr/bin/env python3
"""
دانلود داده‌ی کندل از OKX (فیوچرز دائمی USDT) و ذخیره‌ی دائمی روی دیسک.

- فقط یک تایم‌فریم پایه (پیش‌فرض 15m) دانلود می‌شود؛ تایم‌فریم‌های 1h/4h/1d
  در بک‌تست از روی همین داده ساخته می‌شوند (resample دقیق).
- لیست نمادها مستقیماً از فایل ربات‌ها (SYMBOLS) خوانده می‌شود.
- هر نماد بعد از دانلود در فایل جدا ذخیره می‌شود: data/okx/15m/BTC-USDT-SWAP.csv.gz
- اجرای دوباره، فقط بخش‌های ناقص (قدیمی‌تر یا جدیدتر از فایل موجود) را دانلود می‌کند؛
  اگر همه‌چیز کامل باشد هیچ درخواستی به OKX زده نمی‌شود.
- اگر وسط کار قطع شد، دوباره اجرا کن؛ از همان‌جا ادامه می‌دهد.

مثال:
    python download_okx_data.py                      # ۳۶۵ روز + ۶۰ روز warmup تا امروز (UTC)
    python download_okx_data.py --end 2026-11-17     # پایان ثابت (برای نتایج تکرارپذیر)
    python download_okx_data.py --symbols BTC-SWAP-USDT ETH-SWAP-USDT   # فقط چند نماد
"""
import argparse
import ast
import os
import sys
import time
from datetime import datetime, timezone, timedelta

import pandas as pd
import requests

BASE = "https://www.okx.com"
BAR_MS = {
    "5m": 300_000,     # دقیق‌تر برای تشخیص TP/SL ولی حدود ۳ برابر زمان دانلود
    "15m": 900_000,    # پیش‌فرض
}
COLS = ["ts", "open", "high", "low", "close", "volume"]
MIN_INTERVAL = 0.12          # حدود ۸ درخواست در ثانیه (سقف OKX برای این endpoint: 20 درخواست / 2 ثانیه)

# نمادهایی که در Toobit با پیشوند 1000 هستند ولی در OKX بدون آن معامله می‌شوند.
# (قیمت‌ها فقط یک ضریب ثابت دارند و استراتژی‌ها نسبت به مقیاس بی‌تفاوت‌اند)
OKX_BASE_ALIASES = {
    "1000PEPE": "PEPE", "1000SHIB": "SHIB", "1000BONK": "BONK",
    "1000FLOKI": "FLOKI", "1000SATS": "SATS", "1000RATS": "RATS",
    "1000000MOG": "MOG",
}


def to_okx_inst(symbol):
    """BTC-SWAP-USDT  ->  BTC-USDT-SWAP"""
    base = symbol.split("-")[0]
    base = OKX_BASE_ALIASES.get(base, base)
    return f"{base}-USDT-SWAP"


def load_symbols_from_bots_file(path):
    """SYMBOLS را بدون import کردن فایل ربات‌ها از روی کد می‌خواند."""
    with open(path, "r", encoding="utf-8") as f:
        tree = ast.parse(f.read())
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(
                isinstance(t, ast.Name) and t.id == "SYMBOLS" for t in node.targets):
            return list(ast.literal_eval(node.value))
    raise RuntimeError(f"SYMBOLS در {path} پیدا نشد")


_last_call = [0.0]


def okx_get(session, path, params, retries=8):
    for attempt in range(retries):
        wait = MIN_INTERVAL - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        try:
            r = session.get(BASE + path, params=params, timeout=25)
            if r.status_code == 429:
                time.sleep(2 * (attempt + 1))
                continue
            r.raise_for_status()
            j = r.json()
            code = j.get("code")
            if code == "0":
                return j.get("data", [])
            if code == "50011":            # Too many requests
                time.sleep(2 * (attempt + 1))
                continue
            raise RuntimeError(f"OKX error {code}: {j.get('msg')}")
        except (requests.RequestException, ValueError) as e:
            if attempt == retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError("OKX: تعداد تلاش‌ها تمام شد")


def fetch_range(session, inst, bar, start_ms, end_ms):
    """کندل‌های بسته‌شده با ts در بازه‌ی [start_ms, end_ms) را از جدید به قدیم می‌گیرد."""
    rows = []
    after = end_ms
    while True:
        data = okx_get(session, "/api/v5/market/history-candles",
                       {"instId": inst, "bar": bar, "limit": "100", "after": str(after)})
        if not data:
            break
        for r in data:
            ts = int(r[0])
            if start_ms <= ts < end_ms and r[8] == "1":
                vol = r[6] if r[6] not in ("", None) else r[5]     # volCcy (به واحد کوین)
                rows.append((ts, float(r[1]), float(r[2]), float(r[3]), float(r[4]), float(vol)))
        oldest = int(data[-1][0])
        if oldest <= start_ms or oldest >= after:
            break
        after = oldest
    return rows


def read_cached(path):
    if not os.path.exists(path):
        return pd.DataFrame(columns=COLS)
    try:
        df = pd.read_csv(path)
        return df[COLS]
    except Exception as e:
        print(f"   ⚠️ فایل کش خراب است ({e}) — از نو دانلود می‌شود")
        return pd.DataFrame(columns=COLS)


def save_cached(path, df):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False, compression="gzip")
    os.replace(tmp, path)


def update_symbol(session, inst, bar, start_ms, end_ms, out_dir):
    bar_ms = BAR_MS[bar]
    path = os.path.join(out_dir, bar, f"{inst}.csv.gz")
    old = read_cached(path)
    new_rows = []
    if old.empty:
        new_rows += fetch_range(session, inst, bar, start_ms, end_ms)
    else:
        lo, hi = int(old["ts"].min()), int(old["ts"].max())
        if start_ms < lo:                                   # بخش قدیمی‌تر
            new_rows += fetch_range(session, inst, bar, start_ms, lo)
        if end_ms > hi + bar_ms:                            # بخش جدیدتر
            new_rows += fetch_range(session, inst, bar, hi + bar_ms, end_ms)
    added = len(new_rows)
    if added:
        df = pd.concat([old, pd.DataFrame(new_rows, columns=COLS)], ignore_index=True)
        df = df.drop_duplicates("ts").sort_values("ts").reset_index(drop=True)
        save_cached(path, df)
    else:
        df = old
    return df, added


def describe(df, start_ms, end_ms, bar_ms):
    if df.empty:
        return "بدون داده"
    lo, hi = int(df["ts"].min()), int(df["ts"].max())
    first = max(lo, start_ms)
    expected = (min(hi + bar_ms, end_ms) - first) // bar_ms
    have = int(((df["ts"] >= first) & (df["ts"] < end_ms)).sum())
    cov = 100.0 * have / expected if expected > 0 else 0.0
    d0 = datetime.fromtimestamp(lo / 1000, tz=timezone.utc).strftime("%Y-%m-%d")
    d1 = datetime.fromtimestamp(hi / 1000, tz=timezone.utc).strftime("%Y-%m-%d")
    return f"{d0} → {d1} | {have:,} کندل | پوشش {cov:.2f}%"


def main():
    ap = argparse.ArgumentParser(description="دانلود و کش داده‌ی OKX")
    ap.add_argument("--bots-file", default="crypto_signal_bots_lite.py")
    ap.add_argument("--symbols", nargs="*", default=None, help="لیست نماد (فرمت ربات‌ها). پیش‌فرض: همه")
    ap.add_argument("--bar", default="15m", choices=list(BAR_MS))
    ap.add_argument("--days", type=int, default=365, help="طول بازه‌ی بک‌تست")
    ap.add_argument("--warmup-days", type=int, default=60, help="داده‌ی اضافه قبل از شروع برای محاسبه اندیکاتورها")
    ap.add_argument("--end", default=None, help="تاریخ پایان UTC (YYYY-MM-DD). پیش‌فرض: امروز")
    ap.add_argument("--out", default="data/okx")
    args = ap.parse_args()

    if args.end:
        end_dt = datetime.strptime(args.end, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    else:
        end_dt = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
    start_dt = end_dt - timedelta(days=args.days + args.warmup_days)
    end_ms, start_ms = int(end_dt.timestamp() * 1000), int(start_dt.timestamp() * 1000)
    bar_ms = BAR_MS[args.bar]

    symbols = args.symbols or load_symbols_from_bots_file(args.bots_file)
    print(f"بازه: {start_dt:%Y-%m-%d} → {end_dt:%Y-%m-%d} (UTC) | تایم‌فریم: {args.bar} | {len(symbols)} نماد")
    print(f"پوشه‌ی ذخیره: {os.path.abspath(args.out)}")

    session = requests.Session()
    session.headers["User-Agent"] = "okx-backtest-downloader/1.0"

    listed = None
    try:
        data = okx_get(session, "/api/v5/public/instruments", {"instType": "SWAP"})
        listed = {d["instId"] for d in data if d.get("state") == "live"}
        print(f"OKX: {len(listed)} قرارداد SWAP فعال")
    except Exception as e:
        print(f"⚠️ دریافت لیست قراردادها ناموفق بود ({e}) — همه‌ی نمادها امتحان می‌شوند")

    missing, failed, t0 = [], [], time.time()
    for i, sym in enumerate(symbols, 1):
        inst = to_okx_inst(sym)
        if listed is not None and inst not in listed:
            print(f"[{i}/{len(symbols)}] {inst}: در OKX وجود ندارد — رد شد")
            missing.append(sym)
            continue
        try:
            df, added = update_symbol(session, inst, args.bar, start_ms, end_ms, args.out)
        except KeyboardInterrupt:
            print("\n⏹ متوقف شد. دوباره اجرا کن تا ادامه بدهد (نمادهای کامل‌شده دوباره دانلود نمی‌شوند).")
            sys.exit(1)
        except Exception as e:
            print(f"[{i}/{len(symbols)}] {inst}: ❌ خطا: {e}")
            failed.append(sym)
            continue
        elapsed = time.time() - t0
        eta = elapsed / i * (len(symbols) - i)
        tag = f"+{added:,} جدید" if added else "از کش (بدون دانلود)"
        print(f"[{i}/{len(symbols)}] {inst}: {tag} | {describe(df, start_ms, end_ms, bar_ms)} | ETA {eta/60:.0f}m")

    print("=" * 60)
    if missing:
        print(f"نمادهای موجود نبودن در OKX ({len(missing)}): {', '.join(missing)}")
    if failed:
        print(f"نمادهای ناموفق ({len(failed)}) — دوباره اجرا کن: {', '.join(failed)}")
    print("تمام شد.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile backtest_bots.py
#!/usr/bin/env python3
"""
بک‌تست ۶ ربات روی داده‌ی تاریخی OKX (کش‌شده توسط download_okx_data.py)

روش کار:
  * فایل ربات‌ها (crypto_signal_bots_lite.py) مستقیماً import می‌شود؛ یعنی استراتژی‌ها،
    فیلتر BTC، کولدان، سقف جهت‌ها، مدیریت سرمایه و چرخه‌ی جبران عیناً همان کد زنده‌اند و
    حتی خود تابع run_bot اجرا می‌شود. فقط این‌ها جایگزین شده‌اند:
        دریافت داده / قیمت لحظه‌ای / بررسی TP-SL  ← داده‌ی تاریخی
        تلگرام / اکسل / CSV / فایل JSON            ← حافظه (هیچ چیزی ارسال یا ذخیره نمی‌شود)
        ساعت سیستم                                 ← ساعت شبیه‌سازی
  * تایم‌فریم‌های 1h / 4h / 1d از روی کندل 15m ساخته می‌شوند (UTC).
  * در هر لحظه فقط کندل‌های بسته‌شده دیده می‌شوند (بدون look-ahead).
  * هر ربات هر --step-minutes دقیقه یک‌بار اجرا می‌شود (مثل کرون زنده). قیمت ورود = close آخرین کندل 15m.
  * TP/SL از روی کندل‌های 15m بعد از ورود بررسی می‌شود. اگر در یک کندل هر دو لمس شوند
    (--ambiguous live)، مثل کد زنده نزدیک‌تر بودن close به TP یا SL تعیین می‌کند؛
    با --ambiguous sl حالت محافظه‌کارانه (SL) استفاده می‌شود.

اجرا:
    python backtest_bots.py
    python backtest_bots.py --step-minutes 15 --fee-pct 0.05
    python backtest_bots.py --bots crypto-signal-bot-v70 crypto-signal-bot-v70a
"""
import argparse
import contextlib
import csv
import importlib.util
import io
import json
import os
import sys
import time
import types
from collections import Counter, defaultdict
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import download_okx_data as dl            # فقط برای to_okx_inst و ثابت‌ها

BAR = {"5m": 300_000, "15m": 900_000}
TF_MS = {"15m": 900_000, "1h": 3_600_000, "4h": 14_400_000, "1d": 86_400_000}

STATE = {"t": 0}
STORE = {}            # جایگزین فایل‌های JSON ربات‌ها
TRADES = []           # معاملات بسته‌شده
FUN = defaultdict(Counter)
DATA = {}             # symbol -> tf -> {"df","end_ts"}
RAW = {}              # symbol -> آرایه‌های numpy کندل پایه
STEP_SLICE = {}
STEP_ANALYZE = {}
CFG = {"ambiguous": "live"}
_msg = [0]


# ----------------------------------------------------------------------------
# بارگذاری ربات‌ها با جایگزینی وابستگی‌های بیرونی
# ----------------------------------------------------------------------------
def load_bots_module(path):
    for name in ("ccxt", "jdatetime", "excel_log"):
        sys.modules[name] = types.ModuleType(name)
    sys.modules["excel_log"].log_signal = lambda *a, **k: None
    sys.modules["excel_log"].log_result = lambda *a, **k: None
    spec = importlib.util.spec_from_file_location("bots_lite", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


# ----------------------------------------------------------------------------
# داده
# ----------------------------------------------------------------------------
def resample(df, tf_ms, bar_ms):
    b = (df["ts"] // tf_ms) * tf_ms
    g = df.groupby(b)
    out = g.agg(open=("open", "first"), high=("high", "max"), low=("low", "min"),
                close=("close", "last"), volume=("volume", "sum"), n=("ts", "count"))
    out = out[out["n"] == tf_ms // bar_ms].drop(columns="n")
    out.index.name = "timestamp"
    return out.reset_index()


def load_data(B, symbols, data_dir, bar):
    bar_ms = BAR[bar]
    ok, missing = [], []
    for k, sym in enumerate(symbols, 1):
        inst = dl.to_okx_inst(sym)
        path = os.path.join(data_dir, bar, f"{inst}.csv.gz")
        if not os.path.exists(path):
            missing.append(sym)
            continue
        raw = pd.read_csv(path).drop_duplicates("ts").sort_values("ts").reset_index(drop=True)
        if len(raw) < 100:
            missing.append(sym)
            continue
        RAW[sym] = {"ts": raw["ts"].to_numpy(np.int64), "end": raw["ts"].to_numpy(np.int64) + bar_ms,
                    "high": raw["high"].to_numpy(float), "low": raw["low"].to_numpy(float),
                    "close": raw["close"].to_numpy(float)}
        per_tf = {}
        for tf, ms in TF_MS.items():
            if ms == bar_ms:
                d = raw.rename(columns={"ts": "timestamp"}).copy()
            else:
                d = resample(raw, ms, bar_ms)
            d = B.calculate_indicators(d)
            per_tf[tf] = {"df": d, "end_ts": d["timestamp"].to_numpy(np.int64) + ms}
        DATA[sym] = per_tf
        ok.append(sym)
        if k % 10 == 0:
            print(f"  بارگذاری داده: {k}/{len(symbols)}", flush=True)
    return ok, missing


# ----------------------------------------------------------------------------
# جایگزین‌های توابع ربات
# ----------------------------------------------------------------------------
def p_get_data(ctx, symbol, timeframe):
    key = (symbol, timeframe)
    hit = STEP_SLICE.get(key)
    if hit is not None:
        return hit
    d = DATA[symbol][timeframe]
    n = int(np.searchsorted(d["end_ts"], ctx.ref_time_ms, side="right"))
    limit = 300 if timeframe == "1d" else 150
    if n < 40:
        raise ValueError(f"داده کافی نیست: {symbol} {timeframe}")
    df = d["df"].iloc[max(0, n - limit):n]
    STEP_SLICE[key] = df
    return df


def p_get_price(ctx, symbol):
    return float(p_get_data(ctx, symbol, "15m")["close"].iloc[-1])


def p_check_trade_result(exchange, symbol, data):
    R = RAW[symbol]
    t = STATE["t"]
    start = max(int(data["opened_ms"]), int(data.get("_scan_ms", 0)))
    i0 = int(np.searchsorted(R["ts"], start, side="left"))
    i1 = int(np.searchsorted(R["end"], t, side="right"))
    if i1 <= i0:
        return None, None
    h, l, c = R["high"][i0:i1], R["low"][i0:i1], R["close"][i0:i1]
    tp, sl = data["tp"], data["sl"]
    if data["direction"] == "LONG":
        hs, ht = l <= sl, h >= tp
    else:
        hs, ht = h >= sl, l <= tp
    anyhit = hs | ht
    if not anyhit.any():
        data["_scan_ms"] = int(R["end"][i1 - 1])
        return None, None
    j = int(np.argmax(anyhit))
    data["_hit_ms"] = int(R["ts"][i0 + j])
    if hs[j] and ht[j]:
        data["_ambig"] = True
        if CFG["ambiguous"] == "sl":
            return "SL", sl
        return ("TP", tp) if abs(c[j] - tp) < abs(c[j] - sl) else ("SL", sl)
    return ("TP", tp) if ht[j] else ("SL", sl)


def patch_bots(B, symbols):
    B.SYMBOLS = list(symbols)
    B.get_iran_time = lambda: ("", "", "")
    B.send_telegram = lambda *a, **k: (_msg.__setitem__(0, _msg[0] + 1) or _msg[0])
    B.delete_telegram = lambda *a, **k: None
    B.now_ms = lambda: STATE["t"]

    def load_json(path, default):
        if path not in STORE:
            STORE[path] = default
        return STORE[path]

    def save_json(path, data):
        STORE[path] = data

    B.load_json, B.save_json = load_json, save_json
    B.append_trade_csv = lambda bot_name, record: TRADES.append({**record, "bot": bot_name})
    B.get_data, B.get_price, B.check_trade_result = p_get_data, p_get_price, p_check_trade_result

    def collect(bot_name, top=12):
        c = B.FUNNEL.pop(bot_name, None)
        if c:
            FUN[bot_name].update(c)
    B.print_funnel = collect


def wrap_cached(bot):
    """v0a/v60a/v70a همان تحلیل ربات‌های اصلی را دارند؛ نتیجه در هر گام یک‌بار محاسبه می‌شود."""
    orig = bot.analyze

    def cached(symbol, ctx):
        key = (orig, symbol)
        if key not in STEP_ANALYZE:
            STEP_ANALYZE[key] = orig(symbol, ctx)
        return STEP_ANALYZE[key]
    bot.analyze = cached


# ----------------------------------------------------------------------------
# آمار
# ----------------------------------------------------------------------------
def trade_pnl_r(t):
    risk = abs(t["entry"] - t["sl"])
    if risk <= 0:
        return 0.0
    return abs(t["tp"] - t["entry"]) / risk if t["result"] == "TP" else -1.0


def max_drawdown(series, start=0.0):
    peak, dd, cur = start, 0.0, start
    for x in series:
        cur += x
        peak = max(peak, cur)
        dd = max(dd, peak - cur)
    return dd


def ms_to_date(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d")


def bot_stats(bot, trades, fee_pct, blown_at):
    tr = sorted([t for t in trades if t["bot"] == bot.name], key=lambda t: t.get("_hit_ms", t["closed_ms"]))
    st = {"bot": bot.name, "label": bot.label, "trades": len(tr)}
    open_n = sum(1 for k, v in STORE.get(bot.signals_file, {}).items()
                 if not k.startswith("__") and v.get("status") == "OPEN")
    st["open_at_end"] = open_n
    if not tr:
        return st, tr
    r = [trade_pnl_r(t) for t in tr]
    for t, x in zip(tr, r):
        t["pnl_r"] = round(x, 3)
        risk_pct = abs(t["entry"] - t["sl"]) / t["entry"] * 100
        t["pnl_r_net"] = round(x - (2 * fee_pct / risk_pct if risk_pct > 0 else 0), 3)
    rn = [t["pnl_r_net"] for t in tr]
    wins = [x for x in r if x > 0]
    loss = [x for x in r if x < 0]
    hold = [(t.get("_hit_ms", t["closed_ms"]) - t["opened_ms"]) / 3.6e6 for t in tr]
    st.update({
        "wins": len(wins), "losses": len(loss),
        "win_rate_pct": round(100 * len(wins) / len(tr), 1),
        "total_R": round(sum(r), 2), "avg_R": round(sum(r) / len(r), 3),
        "profit_factor": round(sum(wins) / abs(sum(loss)), 2) if loss else None,
        "max_dd_R": round(max_drawdown(r), 2),
        "total_R_after_fees": round(sum(rn), 2), "max_dd_R_after_fees": round(max_drawdown(rn), 2),
        "avg_rr": round(float(np.mean([abs(t["tp"] - t["entry"]) / abs(t["entry"] - t["sl"]) for t in tr])), 2),
        "avg_hold_h": round(float(np.mean(hold)), 1),
        "longs": sum(1 for t in tr if t["direction"] == "LONG"),
        "shorts": sum(1 for t in tr if t["direction"] == "SHORT"),
        "ambiguous_candles": sum(1 for t in tr if t.get("_ambig")),
        "first_trade": ms_to_date(tr[0]["opened_ms"]), "last_trade": ms_to_date(tr[-1]["opened_ms"]),
    })
    # بدترین رشته‌ی ضرر پشت‌سرهم
    streak = best = 0
    for x in r:
        streak = streak + 1 if x < 0 else 0
        best = max(best, streak)
    st["max_loss_streak"] = best
    if bot.has_sizing:
        bal = [bot.initial_balance] + [t["balance_after"] for t in tr if "balance_after" in t]
        peak, dd = bal[0], 0.0
        for b in bal:
            peak = max(peak, b)
            dd = max(dd, (peak - b) / peak * 100 if peak > 0 else 100.0)
        st.update({
            "initial_balance": bot.initial_balance, "final_balance": round(bal[-1], 2),
            "return_pct": round((bal[-1] / bot.initial_balance - 1) * 100, 1),
            "min_balance": round(min(bal), 2), "max_dd_pct": round(dd, 1),
            "max_leverage": max([float(t.get("leverage", 1) or 1) for t in tr]),
            "blown": blown_at is not None, "blown_date": ms_to_date(blown_at) if blown_at else None,
        })
    # ماهانه
    monthly = defaultdict(float)
    for t, x in zip(tr, r):
        monthly[ms_to_date(t.get("_hit_ms", t["closed_ms"]))[:7]] += x
    st["monthly_R"] = {k: round(v, 2) for k, v in sorted(monthly.items())}
    return st, tr


def print_report(stats, funnels):
    print("\n" + "=" * 100)
    print("نتایج بک‌تست")
    print("=" * 100)
    hdr = f"{'bot':<26}{'trades':>7}{'win%':>7}{'totR':>8}{'avgR':>7}{'PF':>6}{'maxDD_R':>9}{'R(fee)':>8}{'hold_h':>8}{'L/S':>9}"
    print(hdr)
    print("-" * len(hdr))
    for s in stats:
        if not s["trades"]:
            print(f"{s['bot']:<26}{0:>7}  (بدون معامله)")
            continue
        print(f"{s['bot']:<26}{s['trades']:>7}{s['win_rate_pct']:>7}{s['total_R']:>8}{s['avg_R']:>7}"
              f"{str(s['profit_factor']):>6}{s['max_dd_R']:>9}{s['total_R_after_fees']:>8}{s['avg_hold_h']:>8}"
              f"{str(s['longs']) + '/' + str(s['shorts']):>9}")
    man = [s for s in stats if "final_balance" in s]
    if man:
        print("\nربات‌های مدیریت سرمایه (بالانس اولیه و ریسک ۳٪ + چرخه‌ی جبران، بدون کارمزد):")
        h2 = f"{'bot':<26}{'start$':>8}{'final$':>10}{'ret%':>9}{'min$':>9}{'maxDD%':>8}{'maxLev':>8}{'lossStreak':>11}  blown"
        print(h2)
        print("-" * len(h2))
        for s in man:
            print(f"{s['bot']:<26}{s['initial_balance']:>8}{s['final_balance']:>10}{s['return_pct']:>9}"
                  f"{s['min_balance']:>9}{s['max_dd_pct']:>8}{s['max_leverage']:>8}{s['max_loss_streak']:>11}  "
                  f"{'YES ' + s['blown_date'] if s['blown'] else 'no'}")
    print("\nR ماهانه:")
    months = sorted({m for s in stats for m in s.get("monthly_R", {})})
    if months:
        print(f"{'month':<9}" + "".join(f"{s['bot'].replace('crypto-signal-bot', 'bot'):>12}" for s in stats))
        for m in months:
            print(f"{m:<9}" + "".join(f"{s.get('monthly_R', {}).get(m, 0):>12}" for s in stats))
    print("\nپرتکرارترین دلایل رد سیگنال (۵ مورد اول، بدون ~skip ها):")
    for name, c in funnels.items():
        top = [(k, v) for k, v in c.most_common() if not k.startswith(("0_", "~skip"))][:5]
        print(f"  {name}: " + " | ".join(f"{k}={v}" for k, v in top))


# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description="بک‌تست ربات‌ها روی داده‌ی OKX")
    ap.add_argument("--bots-file", default=os.path.join(HERE, "crypto_signal_bots_lite.py"))
    ap.add_argument("--data", default=os.path.join(HERE, "data", "okx"))
    ap.add_argument("--bar", default="15m", choices=list(BAR))
    ap.add_argument("--days", type=int, default=365)
    ap.add_argument("--end", default=None, help="پایان بک‌تست (YYYY-MM-DD, UTC). پیش‌فرض: انتهای داده‌ی BTC")
    ap.add_argument("--step-minutes", type=int, default=60, help="فاصله‌ی اجرای ربات‌ها (مضرب ۱۵)")
    ap.add_argument("--bots", nargs="*", default=None, help="فقط این ربات‌ها (نام کامل)")
    ap.add_argument("--fee-pct", type=float, default=0.05, help="کارمزد هر طرف به درصد (فقط برای ستون R(fee))")
    ap.add_argument("--ambiguous", default="live", choices=["live", "sl"])
    ap.add_argument("--out", default=os.path.join(HERE, "backtest_results"))
    args = ap.parse_args()
    CFG["ambiguous"] = args.ambiguous
    bar_ms = BAR[args.bar]
    if args.step_minutes % 15:
        sys.exit("--step-minutes باید مضرب ۱۵ باشد")

    B = load_bots_module(args.bots_file)
    symbols = list(B.SYMBOLS)
    print(f"بارگذاری داده از {args.data} ...")
    t0 = time.time()
    ok, missing = load_data(B, symbols, args.data, args.bar)
    if B.BTC_SYMBOL not in ok:
        sys.exit("داده‌ی BTC پیدا نشد؛ اول download_okx_data.py را اجرا کن.")
    if missing:
        print(f"⚠️ {len(missing)} نماد بدون داده (کنار گذاشته شد): {', '.join(missing)}")
    print(f"{len(ok)} نماد آماده ({time.time() - t0:.0f}s)")

    btc_last = int(RAW[B.BTC_SYMBOL]["end"][-1])
    end_ms = int(datetime.strptime(args.end, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp() * 1000) \
        if args.end else (btc_last // TF_MS["1d"]) * TF_MS["1d"]
    start_ms = end_ms - args.days * TF_MS["1d"]
    btc_first = int(RAW[B.BTC_SYMBOL]["ts"][0])
    if btc_first > start_ms - 30 * TF_MS["1d"]:
        print(f"⚠️ داده‌ی قبل از شروع کمتر از ۳۰ روز است ({ms_to_date(btc_first)}) — اندیکاتورهای ابتدای بازه کم‌دقت‌اند.")
    print(f"بازه‌ی بک‌تست: {ms_to_date(start_ms)} → {ms_to_date(end_ms)} | گام: {args.step_minutes} دقیقه")

    patch_bots(B, ok)
    bots = [b for b in B.LIVE_BOTS if not args.bots or b.name in args.bots]
    if not bots:
        sys.exit("ربات مطابق --bots پیدا نشد")
    for b in bots:
        wrap_cached(b)
    active = list(bots)
    blown = {}

    step_ms = args.step_minutes * 60_000
    steps = list(range(start_ms, end_ms, step_ms))
    print(f"{len(steps):,} گام × {len(bots)} ربات × {len(ok)} نماد ...\n")
    t0 = time.time()
    devnull = io.StringIO()
    for k, t in enumerate(steps, 1):
        STATE["t"] = t
        STEP_SLICE.clear()
        STEP_ANALYZE.clear()
        ctx = B.Ctx(None, ref_time_ms=t)
        with contextlib.redirect_stdout(devnull):
            btc_4h, btc_1h = B.get_btc_trend(ctx)
            ctx.btc_filter = B.get_btc_filter(ctx)
            for bot in list(active):
                try:
                    B.run_bot(bot, ctx, btc_4h, btc_1h)
                except Exception as e:
                    print(f"خطا {bot.name}: {e}", file=sys.stderr)
                if bot.has_sizing and STORE.get(bot.signals_file, {}).get("__balance__", 1) <= 0:
                    blown[bot.name] = t
                    active.remove(bot)
            devnull.seek(0)
            devnull.truncate(0)
        if k % 200 == 0 or k == len(steps):
            el = time.time() - t0
            print(f"  {k:,}/{len(steps):,} | {ms_to_date(t)} | معاملات بسته: {len(TRADES)} | "
                  f"گذشته {el / 60:.1f}m | باقی‌مانده ~{el / k * (len(steps) - k) / 60:.1f}m", flush=True)

    # خروجی
    os.makedirs(args.out, exist_ok=True)
    stats, all_tr = [], []
    for b in bots:
        s, tr = bot_stats(b, TRADES, args.fee_pct, blown.get(b.name))
        stats.append(s)
        all_tr += tr
    print_report(stats, FUN)

    skip = {"telegram_sent", "telegram_message_id", "close_telegram_sent", "close_msg_attempts", "time"}
    keys = ["bot", "symbol", "direction", "setup", "entry", "sl", "tp", "close_price", "result", "pnl_r", "pnl_r_net",
            "opened_ms", "_hit_ms", "closed_ms", "score", "rsi", "vol_ratio", "atr_pct", "risk_pct", "rr",
            "position_size", "position_value", "leverage", "profit_usd", "balance_after", "cycle_status"]
    keys += sorted({k for t in all_tr for k in t} - set(keys) - skip)
    all_tr.sort(key=lambda t: t["opened_ms"])
    with open(os.path.join(args.out, "trades.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(all_tr)
    with open(os.path.join(args.out, "summary.json"), "w", encoding="utf-8") as f:
        json.dump({"period": [ms_to_date(start_ms), ms_to_date(end_ms)], "step_minutes": args.step_minutes,
                   "symbols_used": len(ok), "symbols_missing": missing, "ambiguous_mode": args.ambiguous,
                   "fee_pct_per_side": args.fee_pct, "bots": stats,
                   "funnel": {k: dict(v) for k, v in FUN.items()}}, f, ensure_ascii=False, indent=2)
    print(f"\nذخیره شد: {os.path.join(args.out, 'trades.csv')} و summary.json")


if __name__ == "__main__":
    main()


## ۳) دانلود داده OKX (بار اول ۱۵ تا ۳۰ دقیقه)
اگر قطع شد یا خطا داد، همین سلول را دوباره اجرا کن؛ ادامه می‌دهد. صفحه را باز و گوشی را روشن نگه دار.

In [ ]:
!python download_okx_data.py --end 2026-11-17

## ۴) اجرای بک‌تست (حدود ۱۰ تا ۱۵ دقیقه)

In [ ]:
!python backtest_bots.py

## ۵) دانلود نتایج

In [ ]:
from google.colab import files
files.download('backtest_results/trades.csv')
files.download('backtest_results/summary.json')